In [1]:
import requests
import pandas as pd
import os
import time
from dotenv import load_dotenv 


# === CONFIG ===
BASE_DIR = r"D:\Android_Mobile_App\AndroidProject_5th"
OUTPUT_CSV = os.path.join(BASE_DIR, "Android_Repos_MultiRange.csv")

# === Load GitHub token ===
load_dotenv("All_Tokens.env")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise ValueError("❌ GitHub token not found in .env")

headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}


# === Define star ranges ===
# Adjust ranges to cover low → mid → high popularity tiers
star_ranges = [
    (50, 100),
    (101, 300),
    (301, 600),
    (601, 1000),
    (1001, 5000),
    (5001,10000),
    (10001,20000)
]

# === Other filters ===
base_query = "android language:Kotlin pushed:>2024-01-01"

# === Search and collect ===
PER_PAGE = 100
PAGES = 10  # max 10 pages per range

all_repos = []

for star_min, star_max in star_ranges:
    query = f"{base_query} stars:{star_min}..{star_max}"
    print(f"🔍 Query: {query}")

    for page in range(1, PAGES + 1):
        print(f"  📄 Fetching page {page} ...")
        url = f"https://api.github.com/search/repositories?q={query}&per_page={PER_PAGE}&page={page}"
        r = requests.get(url, headers=headers)
        
        if r.status_code == 403:
            print("⚠️ Rate limit hit. Sleeping 60 seconds...")
            time.sleep(60)
            continue

        r.raise_for_status()
        data = r.json()
        if "items" not in data or len(data["items"]) == 0:
            print(f"  ✅ No more results for this page.")
            break

        for item in data['items']:
            all_repos.append({
                "full_name": item['full_name'],
                "clone_url": item['clone_url'],
                "stars": item['stargazers_count'],
                "pushed_at": item['pushed_at']
            })

        # Be nice to API
        time.sleep(1)

# === Deduplicate ===
df = pd.DataFrame(all_repos).drop_duplicates(subset="full_name").sort_values(by="stars", ascending=False)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Finished! Total unique repos: {len(df)} unique to : \n{OUTPUT_CSV}")

🔍 Query: android language:Kotlin pushed:>2024-01-01 stars:50..100
  📄 Fetching page 1 ...
  📄 Fetching page 2 ...
  📄 Fetching page 3 ...
  📄 Fetching page 4 ...
  📄 Fetching page 5 ...
  📄 Fetching page 6 ...
  📄 Fetching page 7 ...
  📄 Fetching page 8 ...
  📄 Fetching page 9 ...
  ✅ No more results for this page.
🔍 Query: android language:Kotlin pushed:>2024-01-01 stars:101..300
  📄 Fetching page 1 ...
  📄 Fetching page 2 ...
  📄 Fetching page 3 ...
  📄 Fetching page 4 ...
  📄 Fetching page 5 ...
  📄 Fetching page 6 ...
  📄 Fetching page 7 ...
  📄 Fetching page 8 ...
  📄 Fetching page 9 ...
  📄 Fetching page 10 ...
  ✅ No more results for this page.
🔍 Query: android language:Kotlin pushed:>2024-01-01 stars:301..600
  📄 Fetching page 1 ...
  📄 Fetching page 2 ...
  📄 Fetching page 3 ...
  📄 Fetching page 4 ...
  📄 Fetching page 5 ...
  ✅ No more results for this page.
🔍 Query: android language:Kotlin pushed:>2024-01-01 stars:601..1000
  📄 Fetching page 1 ...
  📄 Fetching page 2 ...
  